In [ ]:
# Install necessary libraries if not already installed
# !pip install transformers datasets accelerate evaluate

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
import evaluate
import numpy as np

# 1. Load the SQuAD dataset
# You can choose 'squad' for SQuAD 1.1 or 'squad_v2' for SQuAD 2.0
dataset = load_dataset("squad") 

# 2. Load a pre-trained tokenizer
model_checkpoint = "distilbert-base-uncased" # Example checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 3. Preprocess the data
def preprocess_function(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_answer = answers[i]
        start_char = sample_answer["answer_start"][0]
        end_char = start_char + len(sample_answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not in the context, set its start and end to 0
        if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise, find the start and end token positions of the answer in the context
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

tokenized_squad = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# 4. Load a pre-trained model for question answering
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

# 5. Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_squad["train"],
    eval_dataset=tokenized_squad["validation"],
    tokenizer=tokenizer,
)

# 7. Train the model
trainer.train()

# 8. (Optional) Evaluate the model
# You might need to implement a more sophisticated evaluation metric for QA like SQuAD's official F1/EM
# For a basic check, you can use the built-in evaluation
# results = trainer.evaluate()
# print(results)

